# 📦 Amazon Parcel Defect Detection with YOLOv8-OBB (Train + Pretrained Evaluation)


In [ ]:
# --- 🔧 Step 1: Install YOLOv8 ---
!pip install ultralytics -q

In [1]:
# --- 📥 Step 2: Clone the GitHub Dataset ---
!git clone https://github.com/saikisri97/Optimisation.git

Cloning into 'Optimisation'...
remote: Enumerating objects: 2761, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 2761 (delta 13), reused 13 (delta 6), pack-reused 2738 (from 2)
Receiving objects: 100% (2761/2761), 122.31 MiB | 12.36 MiB/s, done.
Resolving deltas: 100% (300/300), done.
Updating files: 100% (2438/2438), done.


In [2]:
# --- 🛠️ Step 3: Patch data.yaml for local Colab paths ---
import os

yaml_path = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml"
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

with open(yaml_path, "w") as f:
    f.write("""train: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
val: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
test: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
nc: 2
names: ['defect parcel', 'no defect parcel']
""")

OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
# --- 🖼️ Step 4: Show Training Images with Labels (Both Classes) ---
import glob
import matplotlib.pyplot as plt
from PIL import Image
import os

image_dir = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images"
label_dir = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/labels"

def get_class_label(label_file):
    with open(label_file, 'r') as f:
        lines = f.readlines()
    return int(lines[0].split()[0]) if lines else -1

# Categorize images by class 0 or 1
class0_imgs, class1_imgs = [], []
for img_file in sorted(glob.glob(f"{image_dir}/*.jpg")):
    base = os.path.basename(img_file).replace('.jpg', '.txt')
    label_file = os.path.join(label_dir, base)
    if os.path.exists(label_file):
        label = get_class_label(label_file)
        if label == 0 and len(class0_imgs) < 2:
            class0_imgs.append(img_file)
        elif label == 1 and len(class1_imgs) < 2:
            class1_imgs.append(img_file)
    if len(class0_imgs) >= 2 and len(class1_imgs) >= 2:
        break

# Display
plt.figure(figsize=(10, 5))
for i, img_path in enumerate(class0_imgs + class1_imgs):
    img = Image.open(img_path)
    label = "Defect" if i < 2 else "No Defect"
    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.title(label)
    plt.axis("off")
plt.suptitle("🔍 Training Set: Defect vs No Defect")
plt.tight_layout()
plt.show()


In [ ]:
# --- 🚆 Step 5: Train for 1 Epoch (Demo) ---
from ultralytics import YOLO
model = YOLO("yolov8n-obb.pt")

In [ ]:
# ---- This step is just to show the training loop ----

model.train(
    data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    name="parcel_defect_yolo_demo",
    workers=2
)

In [ ]:
# --- 🧠 Step 6: Load Pretrained Checkpoint (This is how the usage is done - For example ChatGPT  - we use the checkpoint on their Servers. ) ---
pretrained_url = "https://github.com/saikisri97/Optimisation/raw/master/For_AI_Lecture/data/parcel_defect_trained_mac.pt"
checkpoint_path = "/content/yolov8n-obb-parcel-defect.pt"
!wget -O {checkpoint_path} {pretrained_url}

model = YOLO(checkpoint_path)


In [ ]:
# --- 📊 Step 7: Evaluate Pretrained Model ---
metrics = model.val(data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml", split='test')
print("Evaluation Metrics:", metrics)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Run inference
results = model.predict(
    source="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/test/images",
    save=True,
    imgsz=640,
    conf=0.05  # 👈 Lower confidence threshold to get more predictions
)

# Load from existing prediction results
predicted_dir = Path("runs/obb/predict")
class_0_img, class_1_img = None, None

# Loop through predictions and find one of each class
for r in results:
    if r.boxes is not None and len(r.boxes) > 0:
        labels = r.boxes.cls.int().tolist()
        image_path = Path(r.path).name
        saved_img = predicted_dir / image_path

        if saved_img.exists():
            if 0 in labels and class_0_img is None:
                class_0_img = saved_img
            if 1 in labels and class_1_img is None:
                class_1_img = saved_img
        if class_0_img and class_1_img:
            break

# --- Plot the results ---
plt.figure(figsize=(10, 5))

if class_0_img:
    img = cv2.imread(str(class_0_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("🟥 Defect Parcel")
    plt.axis("off")

if class_1_img:
    img = cv2.imread(str(class_1_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 2, 2)
    plt.imshow(img)
    plt.title("🟩 No Defect Parcel")
    plt.axis("off")

plt.suptitle("📦 YOLOv8-OBB Inference: One Prediction Per Class", fontsize=16)
plt.tight_layout()
plt.show()
